# SemEval-2026 Task 13 — Subtask B: GraphCodeBERT

**Multi-Class Authorship Detection** — 11 classes (human + 10 LLM families)

### Model: `microsoft/graphcodebert-base`
- Pre-trained with data flow information → structure-aware representations
- Same RoBERTa architecture as CodeBERT → drop-in replacement
- Hypothesis: DFG-aware pretraining captures code structure patterns per generator

### Class Imbalance Handling
- **Sqrt-scaled class-weighted loss** (prevents extreme weight ratios)
- **Stratified subsampling** (cap human class at ~45K, keep all minority → ~10% total)
- **Label smoothing** (α = 0.1)

### Setup
- **GPU:** Kaggle Tesla T4 (16 GB VRAM)
- **Data:** 10% effective via human class capping
- **Metric:** Macro F1 (competition metric)

Set runtime to `GPU T4 x2` or `GPU T4` in Kaggle settings.

In [ ]:
# ============================================================
# Cell 1: Environment Setup
# ============================================================
!pip install --upgrade transformers==4.45.0 huggingface_hub datasets scikit-learn accelerate -q

## Data Setup
Uncomment the option matching your environment.

In [ ]:
# ============================================================
# Cell 2: Data paths
# ============================================================

# --- OPTION A: Kaggle input paths ---
# TRAIN_PATH = "/kaggle/input/datasets/daniilor/semeval-2026-task13/SemEval-2026-Task13/task_b/task_b_training_set.parquet"
# VAL_PATH   = "/kaggle/input/datasets/daniilor/semeval-2026-task13/SemEval-2026-Task13/task_b/task_b_validation_set.parquet"
# TEST_PATH  = "/kaggle/input/datasets/daniilor/semeval-2026-task13/SemEval-2026-Task13/task_b/task_b_test_set_sample.parquet"

# --- OPTION B: Download from HuggingFace (default) ---
from datasets import load_dataset
print("Downloading SemEval-2026-Task13 Subtask B from HuggingFace...")
hf_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "B")
hf_dataset['train'].to_parquet('task_b_train.parquet')
hf_dataset['validation'].to_parquet('task_b_val.parquet')
if 'test' in hf_dataset:
    hf_dataset['test'].to_parquet('task_b_test.parquet')
    TEST_PATH = 'task_b_test.parquet'
else:
    TEST_PATH = None
    print("No test split available on HuggingFace.")
TRAIN_PATH = 'task_b_train.parquet'
VAL_PATH   = 'task_b_val.parquet'
print("Done!")

print(f"Train: {TRAIN_PATH}")
print(f"Val:   {VAL_PATH}")
print(f"Test:  {TEST_PATH}")

## Configuration

In [ ]:
# ============================================================
# Cell 3: Configuration
# ============================================================

# --- Model ---
MODEL_NAME = "microsoft/graphcodebert-base"

# --- Subsampling (target: 10% of ~500K = ~50K total) ---
# Strategy: cap human class at 45K, keep ALL minority (~5K)
# Result:   45K + 5K = ~50K ≈ 10% of full dataset
USE_SUBSET        = True
HUMAN_SUBSET_SIZE = 45000    # Cap human class (originally ~450K)
VAL_FRACTION      = 0.1      # Keep 10% of validation
RANDOM_SEED       = 42

# --- Training hyperparameters ---
MAX_LENGTH         = 256     # Sequence length (T4-safe)
BATCH_SIZE         = 16
GRAD_ACCUM_STEPS   = 2       # Effective batch = 32
LEARNING_RATE      = 2e-5
NUM_EPOCHS         = 10
WEIGHT_DECAY       = 0.01
WARMUP_RATIO       = 0.1
LABEL_SMOOTHING    = 0.1     # Smoothing for class-weighted loss
EARLY_STOPPING_PATIENCE = 3

# --- Output ---
OUTPUT_DIR = "/kaggle/working/results_graphcodebert_taskB"

# --- Label mapping ---
ID_TO_LABEL = {
    0: "human", 1: "deepseek", 2: "qwen", 3: "01-ai", 4: "bigcode",
    5: "gemma", 6: "phi", 7: "meta-llama", 8: "ibm-granite", 9: "mistral", 10: "openai"
}
LABEL_TO_ID = {v: k for k, v in ID_TO_LABEL.items()}
NUM_LABELS = len(ID_TO_LABEL)

print(f"Model: {MODEL_NAME}")
print(f"Task B: {NUM_LABELS}-class classification")
print(f"Labels: {list(ID_TO_LABEL.values())}")
print(f"Subset mode: {USE_SUBSET}")
print(f"Human cap: {HUMAN_SUBSET_SIZE} (target ~10% total with minorities)")
print(f"Max length: {MAX_LENGTH}")
print(f"Label smoothing: {LABEL_SMOOTHING}")

In [ ]:
# ============================================================
# Cell 4: Imports
# ============================================================

import os
os.environ["WANDB_DISABLED"] = "true"

import gc
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from datasets import Dataset
import transformers
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    f1_score,
)
import warnings
warnings.filterwarnings("ignore")

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f"VRAM: {vram / 1e9:.1f} GB")

## Class Imbalance Handling

- **Stratified subsampling**: Cap the human class at 45K, keep ALL minority samples (~5K) → ~50K total (10%)
- **Sqrt-scaled class weights**: Prevents extreme weight ratios (raw ~220:1 → √ ~15:1)
- **Label smoothing**: α=0.1 in the loss function

In [ ]:
# ============================================================
# Cell 5: WeightedTrainer + utilities (from improved baseline)
# ============================================================

class WeightedTrainer(Trainer):
    """
    Custom Trainer with:
    - Sqrt-scaled class-weighted CrossEntropyLoss
    - Label smoothing (α ≈ 0.1)
    """

    def __init__(self, class_weights=None, label_smoothing=0.1, **kwargs):
        super().__init__(**kwargs)
        self.label_smoothing = label_smoothing
        if class_weights is not None:
            self.class_weights = class_weights.to(self.model.device)
        else:
            self.class_weights = None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
            loss_fn = nn.CrossEntropyLoss(
                weight=weights,
                label_smoothing=self.label_smoothing,
            )
        else:
            loss_fn = nn.CrossEntropyLoss(
                label_smoothing=self.label_smoothing,
            )

        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


def stratified_subsample(df, human_subset_size=HUMAN_SUBSET_SIZE,
                         random_state=RANDOM_SEED):
    """Keep ALL minority samples, downsample human class to get ~10% total."""
    human_df = df[df['label'] == 0]
    minority_df = df[df['label'] != 0]

    original_total = len(df)
    if len(human_df) > human_subset_size:
        human_df = human_df.sample(n=human_subset_size, random_state=random_state)

    result = pd.concat([human_df, minority_df], ignore_index=True)
    result = result.sample(frac=1, random_state=random_state).reset_index(drop=True)

    pct = len(result) / original_total * 100
    print(f"  Subsampled: {original_total} -> {len(result)} samples ({pct:.1f}%)")
    print(f"  Human: {len(human_df)} | Minority (all 10 classes): {len(minority_df)}")
    return result


def compute_class_weights(label_counts, num_labels, method="sqrt"):
    """
    Compute class weights with sqrt scaling.
    Raw inverse frequency: 220:1 ratio -> unstable.
    sqrt scaling: ~15:1 ratio -> stable.
    """
    total = sum(label_counts.get(i, 1) for i in range(num_labels))
    weights = []
    for i in range(num_labels):
        count = label_counts.get(i, 1)
        raw_weight = total / (num_labels * count)
        if method == "sqrt":
            w = np.sqrt(raw_weight)
        elif method == "log":
            w = np.log1p(raw_weight)
        else:
            w = raw_weight
        weights.append(w)

    min_w = min(weights)
    weights = [w / min_w for w in weights]
    return torch.FloatTensor(weights)


print("WeightedTrainer, stratified_subsample(), compute_class_weights() defined.")

## GraphCodeBERT Trainer for Task B

Follows the baseline `ImprovedCodeTrainer` structure with:
- GraphCodeBERT backbone
- Sqrt-scaled class-weighted loss + label smoothing
- Macro F1 as primary metric

In [ ]:
# ============================================================
# Cell 6: GraphCodeBERT Trainer for Task B
# ============================================================

class GraphCodeBERTTrainerB:
    """
    Task B trainer using GraphCodeBERT.
    Mirrors the baseline ImprovedCodeTrainer pipeline:
      load_and_prepare_data → initialize_model_and_tokenizer →
      prepare_datasets → train → evaluate_model

    Key additions over Task A:
      - Sqrt-scaled class-weighted CrossEntropyLoss
      - Stratified subsampling (cap human class → 10% total)
      - Label smoothing (α = 0.1)
    """

    def __init__(self, model_name=MODEL_NAME, max_length=MAX_LENGTH):
        self.model_name = model_name
        self.max_length = max_length
        self.tokenizer = None
        self.model = None
        self.class_weights = None
        self.tag = model_name.split("/")[-1]

    # ------------------------------------------------------------------
    # Data loading + stratified subsampling (10%)
    # ------------------------------------------------------------------
    def load_and_prepare_data(self):
        try:
            df = pd.read_parquet(TRAIN_PATH)
            print(f"[{self.tag}] Dataset columns: {df.columns.tolist()}")
            print(f"[{self.tag}] Original dataset size: {len(df)}")

            if 'code' not in df.columns or 'label' not in df.columns:
                raise ValueError("Dataset must contain 'code' and 'label' columns")

            df = df.dropna(subset=['code', 'label'])
            df['label'] = df['label'].astype(int)

            unique_labels = sorted(df['label'].unique())
            print(f"\n[{self.tag}] Unique labels: {unique_labels}")

            # Full dataset distribution
            print(f"\n[{self.tag}] Full dataset label distribution:")
            label_counts = df['label'].value_counts().sort_index()
            for label_id, count in label_counts.items():
                name = ID_TO_LABEL.get(label_id, f"unknown-{label_id}")
                print(f"  {label_id} ({name:>12s}): {count:>7d} ({count/len(df)*100:.1f}%)")

            # --- Stratified subsampling: cap human class → ~10% total ---
            if USE_SUBSET:
                print(f"\n--- SUBSET MODE (target: 10% of dataset) ---")
                df = stratified_subsample(df, human_subset_size=HUMAN_SUBSET_SIZE)

            # --- Compute sqrt-scaled class weights ---
            label_counts = df['label'].value_counts().sort_index()
            self.class_weights = compute_class_weights(
                label_counts, NUM_LABELS, method="sqrt"
            )

            print(f"\n[{self.tag}] Class weights (sqrt-scaled):")
            for i, w in enumerate(self.class_weights.tolist()):
                name = ID_TO_LABEL.get(i, f"unknown-{i}")
                count = label_counts.get(i, 0)
                print(f"  {name:>12s}: weight={w:>5.2f}  (n={count})")

            # --- Validation set (10% stratified) ---
            val_df = pd.read_parquet(VAL_PATH)
            val_df['label'] = val_df['label'].astype(int)

            if USE_SUBSET:
                print(f"\n--- Subsampling validation to {VAL_FRACTION*100:.0f}% ---")
                val_df = val_df.groupby('label', group_keys=False).apply(
                    lambda x: x.sample(frac=VAL_FRACTION, random_state=RANDOM_SEED)
                ).reset_index(drop=True)

            print(f"\n[{self.tag}] Final -> Train: {len(df)} | Val: {len(val_df)}")
            return df, val_df

        except Exception as e:
            print(f"[{self.tag}] Error loading dataset: {e}")
            raise

    # ------------------------------------------------------------------
    # Model + tokenizer initialization
    # ------------------------------------------------------------------
    def initialize_model_and_tokenizer(self):
        print(f"\n[{self.tag}] Initialising model and tokenizer ...")

        # GraphCodeBERT uses the same RobertaTokenizer as CodeBERT
        self.tokenizer = RobertaTokenizer.from_pretrained(self.model_name)

        # Standard sequence classification head
        self.model = RobertaForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=NUM_LABELS,
            problem_type="single_label_classification",
        ).to('cuda')

        total_params = sum(p.numel() for p in self.model.parameters())
        train_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"[{self.tag}] {NUM_LABELS} labels | params {total_params:,} | trainable {train_params:,}")

    # ------------------------------------------------------------------
    # Tokenization
    # ------------------------------------------------------------------
    def tokenize_function(self, examples):
        return self.tokenizer(
            examples['code'],
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

    # ------------------------------------------------------------------
    # Dataset preparation
    # ------------------------------------------------------------------
    def prepare_datasets(self, train_df, val_df):
        print(f"\n[{self.tag}] Tokenizing datasets...")
        train_dataset = Dataset.from_pandas(train_df[['code', 'label']])
        val_dataset   = Dataset.from_pandas(val_df[['code', 'label']])

        train_dataset = train_dataset.map(
            self.tokenize_function, batched=True, remove_columns=['code']
        )
        val_dataset = val_dataset.map(
            self.tokenize_function, batched=True, remove_columns=['code']
        )

        train_dataset = train_dataset.rename_column('label', 'labels')
        val_dataset   = val_dataset.rename_column('label', 'labels')

        print(f"[{self.tag}] Done. Train: {len(train_dataset)}, Val: {len(val_dataset)}")
        return train_dataset, val_dataset

    # ------------------------------------------------------------------
    # Metrics — Macro F1 (competition metric)
    # ------------------------------------------------------------------
    def compute_metrics(self, eval_pred):
        predictions, labels = eval_pred
        preds = np.argmax(predictions, axis=1)

        # Log prediction distribution for debugging
        unique, counts = np.unique(preds, return_counts=True)
        pred_dist = {int(u): int(c) for u, c in zip(unique, counts)}
        print(f"  Prediction distribution: {pred_dist}")

        accuracy = accuracy_score(labels, preds)
        macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
        weighted_f1 = f1_score(labels, preds, average='weighted', zero_division=0)

        return {
            'macro_f1': macro_f1,
            'weighted_f1': weighted_f1,
            'accuracy': accuracy,
        }

    # ------------------------------------------------------------------
    # Training (with WeightedTrainer)
    # ------------------------------------------------------------------
    def train(
        self,
        train_dataset,
        val_dataset,
        output_dir=OUTPUT_DIR,
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
    ):
        print(f"\n{'='*60}")
        print(f"  STARTING TRAINING — {self.tag}")
        print(f"{'='*60}")

        steps_per_epoch = len(train_dataset) // (batch_size * GRAD_ACCUM_STEPS)
        eval_save_steps = max(200, steps_per_epoch // 3)

        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size * 2,
            gradient_accumulation_steps=GRAD_ACCUM_STEPS,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            logging_dir=f"{output_dir}/logs",
            logging_steps=50,
            eval_strategy="steps",
            eval_steps=eval_save_steps,
            save_strategy="steps",
            save_steps=eval_save_steps,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            remove_unused_columns=False,
            learning_rate=learning_rate,
            lr_scheduler_type="cosine",
            save_total_limit=2,
            fp16=True,
            report_to="none",
        )

        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

        # Use WeightedTrainer with class weights + label smoothing
        trainer = WeightedTrainer(
            class_weights=self.class_weights,
            label_smoothing=LABEL_SMOOTHING,
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=self.tokenizer,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE
            )],
        )

        print(f"Model: {self.model_name}")
        print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
        print(f"Batch: {batch_size}x{GRAD_ACCUM_STEPS}, LR: {learning_rate}, MaxLen: {self.max_length}")
        print(f"Epochs: {num_epochs}, Eval every {eval_save_steps} steps")
        print(f"fp16: True, Scheduler: cosine")
        print(f"Class weights: sqrt-scaled, Label smoothing: {LABEL_SMOOTHING}")
        print()

        trainer.train()

        trainer.save_model()
        self.tokenizer.save_pretrained(output_dir)
        print(f"\n[{self.tag}] Training completed. Model saved to {output_dir}")
        return trainer

    # ------------------------------------------------------------------
    # Evaluation
    # ------------------------------------------------------------------
    def evaluate_model(self, trainer, val_dataset):
        print(f"\n{'='*60}")
        print(f"  EVALUATION — {self.tag}")
        print(f"{'='*60}")

        predictions = trainer.predict(val_dataset)
        y_pred = np.argmax(predictions.predictions, axis=1)
        y_true = predictions.label_ids

        # Prediction distribution
        unique, counts = np.unique(y_pred, return_counts=True)
        print("\nPrediction distribution:")
        for u, c in zip(unique, counts):
            name = ID_TO_LABEL.get(int(u), f"unknown-{u}")
            print(f"  Predicted {name:>12s} (label {u}): {c} times")

        # Per-class classification report
        target_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
        print("\nPer-Class Classification Report:")
        print(classification_report(
            y_true, y_pred,
            target_names=target_names,
            zero_division=0,
        ))

        # Competition metric
        macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        print(f"\n>>> COMPETITION METRIC (Macro F1): {macro_f1:.4f} <<<")
        return predictions

    # ------------------------------------------------------------------
    # Full pipeline
    # ------------------------------------------------------------------
    def run_full_pipeline(
        self,
        output_dir=OUTPUT_DIR,
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
    ):
        try:
            train_df, val_df = self.load_and_prepare_data()
            self.initialize_model_and_tokenizer()
            train_dataset, val_dataset = self.prepare_datasets(train_df, val_df)
            trainer = self.train(
                train_dataset, val_dataset,
                output_dir=output_dir,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
            )
            self.evaluate_model(trainer, val_dataset)
            print(f"\n[{self.tag}] Pipeline completed successfully!")
            return trainer
        except Exception as e:
            print(f"[{self.tag}] Error in pipeline: {e}")
            raise

print("GraphCodeBERTTrainerB defined.")

---
## Run Training + Evaluation

In [ ]:
# ============================================================
# Cell 7: Run GraphCodeBERT on Task B
# ============================================================

print("\n" + "=" * 70)
print("  GraphCodeBERT — Task B (11-Class Authorship Detection)")
print("=" * 70)

trainer_obj = GraphCodeBERTTrainerB(
    model_name=MODEL_NAME,
    max_length=MAX_LENGTH,
)

hf_trainer = trainer_obj.run_full_pipeline(
    output_dir=OUTPUT_DIR,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
)

---
## Predict on Test Set

In [ ]:
# ============================================================
# Cell 8: Test set prediction
# ============================================================
from itertools import chain
from tqdm import tqdm


@torch.no_grad()
def predict_with_trainer(trainer_obj, parquet_path, output_path,
                         max_length=256, batch_size=16, device=None):
    """
    Run streaming inference over a parquet file.
    Writes 'ID,prediction' CSV.
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = trainer_obj.model
    tokenizer = trainer_obj.tokenizer
    if tokenizer is None:
        raise ValueError("trainer_obj must have a tokenizer.")

    model.to(device)
    model.eval()

    from datasets import load_dataset
    ds = load_dataset("parquet", data_files=parquet_path, split="train", streaming=True)

    it = iter(ds)
    first = next(it)
    id_col = "ID" if "ID" in first else "id" if "id" in first else None
    if not id_col or "code" not in first:
        raise ValueError("Parquet must contain 'ID' (or 'id') and 'code' columns")
    stream = chain([first], it)

    def batcher(iterator, bs):
        buf = []
        for ex in iterator:
            buf.append(ex)
            if len(buf) == bs:
                yield buf
                buf = []
        if buf:
            yield buf

    with open(output_path, "w") as f:
        f.write("ID,prediction\n")
        for batch in tqdm(batcher(stream, batch_size), desc="Predicting"):
            codes = [row["code"] for row in batch]
            ids   = [row[id_col] for row in batch]
            enc = tokenizer(
                codes, truncation=True, padding=True,
                max_length=max_length, return_tensors="pt",
            )
            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            pred_labels = logits.argmax(dim=-1).cpu().tolist()
            for ex_id, pred in zip(ids, pred_labels):
                f.write(f"{ex_id},{pred}\n")

    print(f"Predictions saved to {output_path}")

print("predict_with_trainer() defined.")

In [ ]:
# ============================================================
# Cell 9: Generate submission
# ============================================================

if TEST_PATH is not None:
    OUT_CSV = "submission_b_graphcodebert.csv"
    predict_with_trainer(
        trainer_obj=trainer_obj,
        parquet_path=TEST_PATH,
        output_path=OUT_CSV,
        max_length=MAX_LENGTH,
        batch_size=32,
        device="cuda",
    )
    print("Wrote:", OUT_CSV)
else:
    print("No test path set. Update TEST_PATH in the data setup cell.")

In [ ]:
# ============================================================
# Cell 10: Clean up GPU memory
# ============================================================

del hf_trainer, trainer_obj
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed.")